# Retail Store Turnaround — Data Cleaning

Simple cleaning of the raw data with **pandas**: remove duplicate transactions and handle missing values,
then export clean CSV files for SQL Server. Only the `transactions` table needs cleaning — the three
dimension tables (Stores, Products, Customers) were already clean.

In [1]:
import pandas as pd

# load the four raw files
stores       = pd.read_csv("../data/raw/Stores.csv")
products     = pd.read_csv("../data/raw/Products.csv")
customers    = pd.read_csv("../data/raw/Customers.csv")
transactions = pd.read_csv("../data/raw/transactions.csv")

print("transactions:", transactions.shape)
transactions.head()

transactions: (250500, 12)


,Transaction_ID,Date,Store_ID,Product_ID,Customer_ID,Quantity,Unit_Price,Discount_Percent,Revenue,Cost,Profit,Payment_Method
0,1,2023-04-18,35,179,13345,1,1401.52,29.35,990.17,917.11,73.06,UPI
1,2,2024-11-11,44,107,5232,1,3066.08,21.04,2420.98,2286.23,134.75,UPI
2,3,2025-06-12,100,180,19583,2,5134.90,23.03,7904.67,7813.57,91.10,UPI
3,4,2023-03-31,12,292,4137,3,4249.26,14.08,10952.89,8382.11,2570.78,Credit Card
4,5,2024-06-27,61,76,9998,1,1984.15,12.01,1745.85,1535.81,210.04,Credit Card


## 1. Check missing values and duplicates

In [2]:
print("Missing values per column:")
print(transactions.isna().sum())

print("\nDuplicate rows          :", transactions.duplicated().sum())
print("Duplicate Transaction_IDs:", transactions["Transaction_ID"].duplicated().sum())

Missing values per column:
Transaction_ID        0
Date                  0
Store_ID              0
Product_ID            0
Customer_ID           0
Quantity              0
Unit_Price            0
Discount_Percent    250
Revenue               0
Cost                  0
Profit                0
Payment_Method      250
dtype: int64



Duplicate rows          : 498
Duplicate Transaction_IDs: 500


## 2. Remove duplicates

In [3]:
before = len(transactions)

transactions = transactions.drop_duplicates()                        # drop exact duplicate rows
transactions = transactions.drop_duplicates(subset="Transaction_ID") # keep one row per Transaction_ID

print("Removed", before - len(transactions), "duplicate rows")
print("Rows now :", len(transactions))

Removed 500 duplicate rows
Rows now : 250000


## 3. Handle missing values

In [4]:
# Discount_Percent (numeric): fill missing with the median discount
transactions["Discount_Percent"] = transactions["Discount_Percent"].fillna(
    transactions["Discount_Percent"].median())

# Payment_Method (text): fill missing with "Unknown"
transactions["Payment_Method"] = transactions["Payment_Method"].fillna("Unknown")

print("Missing values remaining:", transactions.isna().sum().sum())

Missing values remaining: 0


## 4. Fix the date column

In [5]:
transactions["Date"] = pd.to_datetime(transactions["Date"])
print(transactions.dtypes)

Transaction_ID               int64
Date                datetime64[us]
Store_ID                     int64
Product_ID                   int64
Customer_ID                  int64
Quantity                     int64
Unit_Price                 float64
Discount_Percent           float64
Revenue                    float64
Cost                       float64
Profit                     float64
Payment_Method                 str
dtype: object


## 5. Export the clean files

In [6]:
transactions.to_csv("../data/clean/transactions_clean.csv", index=False)
stores.to_csv("../data/clean/stores_clean.csv", index=False)
products.to_csv("../data/clean/products_clean.csv", index=False)
customers.to_csv("../data/clean/customers_clean.csv", index=False)

print("Saved cleaned files to ../data/clean/")

Saved cleaned files to ../data/clean/
